# DST Assignment 04 – Machine Learning Pipeline

In this assignment, you will build a complete Machine Learning classification pipeline. The dataset contains 10,000 student records.

**Core Task:** Implement the K-Nearest Neighbors (KNN) algorithm entirely from scratch using only NumPy. Then, compare your implementation against `scikit-learn` models.

> **Rules:**
> - You must use **exactly** the variable names requested in the `# TODO:` comments.
> - You may NOT use `sklearn.neighbors` or any other library for Task 4.
> - For random states, always use `random_state=42` to ensure your results match the autograder.

## Task 1: Data Loading & Exploration

In [18]:
# Download the dataset
# The previous wget command resulted in an empty file. Trying a different, known working source.
!wget -q https://raw.githubusercontent.com/plotly/datasets/master/students_performance.csv -O students_performance.csv

# If this link also doesn't work, you might need to upload the file manually
# or find an alternative public source and update the wget URL.

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# TODO: Load 'students_performance.csv' into `df`
df = pd.read_csv('/content/students_performance (2).csv')
# TODO: Get the shape of the dataframe into `df_shape`
df_shape = df.shape

# TODO: Create a Series of missing value counts per column in `missing_counts`
missing_counts = df.isnull().sum()

# TODO: Create a Series containing the percentage distribution of the target 'grade' in `class_distribution`
class_distribution = df['grade'].value_counts(normalize=True) * 100

## Task 2: Preprocessing

In [20]:
# TODO: Create a copy of df called `df_clean`
df_clean = df.copy()

# TODO: Fill missing values in 'previous_gpa' and 'sleep_hours' with the MEDIAN of their respective columns
df_clean['previous_gpa'] = df_clean['previous_gpa'].fillna(df_clean['previous_gpa'].median())
df_clean['sleep_hours'] = df_clean['sleep_hours'].fillna(df_clean['sleep_hours'].median())

# TODO: Create `label_map` dict to map 'extracurricular' ('Yes' -> 1, 'No' -> 0)
label_map = {'Yes': 1, 'No': 0}
# Apply the map to df_clean['extracurricular']
df_clean['extracurricular'] = df_clean['extracurricular'].map(label_map)

# TODO: Drop the 'student_id' column (it should not be a feature)
df_clean = df_clean.drop(columns=['student_id'])

# TODO: Separate features into `X` (DataFrame) and target into `y` (Series)
X = df_clean.drop(columns=['grade'])
y = df_clean['grade']

## Task 3: Feature Scaling & Train/Test Split

In [21]:
from sklearn.model_selection import train_test_split

# TODO: Implement manual standard scaling: Z = (X - mean) / std
# Calculate the mean and std for each column in X using NumPy or Pandas.
# Store the means in `scaler_mean` (Series or array) and stds in `scaler_std` (Series or array)
scaler_mean = X.mean()
scaler_std = X.std()

# TODO: Create `X_scaled` (DataFrame or numpy array) using your calculated mean/std
X_scaled = (X - scaler_mean) / scaler_std

# TODO: Split X_scaled and y into train and test sets (80/20 split, random_state=42)
# Use variables: X_train, X_test, y_train, y_test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

## Task 4: KNN From Scratch (3 pts)
Do NOT use sklearn for this cell. Use pure NumPy.

In [22]:
def knn_predict(X_train_data, y_train_data, X_test_data, k=5):
    """
    Implement KNN classification.
    """
    predictions = []

    for test_row in X_test_data:
        # 1. Compute Euclidean distances to all training samples
        distances = np.sqrt(np.sum((X_train_data - test_row) ** 2, axis=1))

        # 2. Find the indices of the k nearest neighbors
        # argpartition splits indices so the smallest k values are in the front
        k_indices = np.argpartition(distances, k)[:k]
        # Sort them by actual distance to ensure stable index tie-breaks
        k_indices = k_indices[np.argsort(distances[k_indices])]

        # 3. Get the classes of those k neighbors
        k_nearest_labels = y_train_data[k_indices]

        # 4. Perform a majority vote to pick the predicted class
        labels, counts = np.unique(k_nearest_labels, return_counts=True)
        max_count = np.max(counts)

        # Handle potential ties
        winners = labels[counts == max_count]
        if len(winners) > 1:
            # Tie-break votes by picking the alphabetically smallest label
            winners = np.sort(winners)

        predictions.append(winners[0])

    return np.array(predictions)

# Convert to numpy arrays if they aren't already
X_train_np = np.array(X_train)
y_train_np = np.array(y_train)
X_test_np = np.array(X_test)
y_test_np = np.array(y_test)

# TODO: Use your function to predict with k=5, store in `knn_preds_k5`
knn_preds_k5 = knn_predict(X_train_np, y_train_np, X_test_np, k=5)

# TODO: Calculate accuracy for k=5, store in `knn_accuracy_k5`
knn_accuracy_k5 = np.mean(knn_preds_k5 == y_test_np)

# TODO: Use your function to predict with k=3, store in `knn_preds_k3`
knn_preds_k3 = knn_predict(X_train_np, y_train_np, X_test_np, k=3)

# TODO: Calculate accuracy for k=3, store in `knn_accuracy_k3`
knn_accuracy_k3 = np.mean(knn_preds_k3 == y_test_np)

## Task 5: Scikit-Learn Models
Compare your implementation against established algorithms.

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# TODO: Train Logistic Regression (max_iter=1000, random_state=42)
# Store model in `logreg_model`, accuracy in `logreg_accuracy`
logreg_model = LogisticRegression(max_iter=1000, random_state=42)
logreg_model.fit(X_train, y_train)
logreg_accuracy = accuracy_score(y_test, logreg_model.predict(X_test))

# TODO: Train Decision Tree (random_state=42)
# Store model in `dt_model`, accuracy in `dt_accuracy`
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_accuracy = accuracy_score(y_test, dt_model.predict(X_test))

# TODO: Train Random Forest (n_estimators=100, random_state=42)
# Store model in `rf_model`, accuracy in `rf_accuracy`
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_accuracy = accuracy_score(y_test, rf_model.predict(X_test))

## Task 6: Evaluation

In [24]:
from sklearn.metrics import confusion_matrix, classification_report

# Dictionary to hold accuracy values mapped to assignment-requested model names
accuracies = {
    'KNN': knn_accuracy_k5,
    'LogisticRegression': logreg_accuracy,
    'DecisionTree': dt_accuracy,
    'RandomForest': rf_accuracy
}

# TODO: Determine which model had the best accuracy (from KNN-k5, LogReg, DT, RF)
# Set `best_model_name` to one of: 'KNN', 'LogisticRegression', 'DecisionTree', 'RandomForest'
best_model_name = max(accuracies, key=accuracies.get)

# Setup best predictions mapping dynamically based on the best performing architecture
preds_map = {
    'KNN': knn_preds_k5,
    'LogisticRegression': logreg_model.predict(X_test),
    'DecisionTree': dt_model.predict(X_test),
    'RandomForest': rf_model.predict(X_test)
}
best_preds = preds_map[best_model_name]

# TODO: Generate the confusion matrix for the BEST model's predictions on X_test
confusion_mat = confusion_matrix(y_test, best_preds)

# TODO: Generate the classification report as a DICTIONARY for the BEST model
# Use output_dict=True
class_report = classification_report(y_test, best_preds, output_dict=True)

## Task 7: Model Comparison & Cross-Validation

In [25]:
from sklearn.model_selection import cross_val_score

# TODO: Create a DataFrame `comparison_df` with columns: ['Model', 'Accuracy']
# It should contain rows for 'KNN-k5', 'LogisticRegression', 'DecisionTree', and 'RandomForest'
comparison_df = pd.DataFrame({
    'Model': ['KNN-k5', 'LogisticRegression', 'DecisionTree', 'RandomForest'],
    'Accuracy': [knn_accuracy_k5, logreg_accuracy, dt_accuracy, rf_accuracy]
})

# TODO: Run 5-fold cross-validation on the Random Forest model using the FULL scaled dataset (X_scaled, y)
# Store the array of 5 scores in `cv_scores`
cv_scores = cross_val_score(rf_model, X_scaled, y, cv=5)